# Solution: Build a Flights CodeAct Agent with DSPy

Answer key for [flights_code_agent.ipynb](flights_code_agent.ipynb).

You already taught an agent to book flights by picking tools one JSON call at a time — what if it could write a short Python script instead?

In the [Flights Agent (ReAct)](flights_agent.ipynb) exercise, the LM chose **one tool and its arguments** per step. In this exercise, you rebuild the same airline customer service agent using [**CodeAct**](https://dspy.ai/api/modules/CodeAct/) from [lesson 10](../lessons/11_dspy_CodeAct.ipynb): the LM writes **executable Python** that orchestrates your tools inside a code interpreter.

**Prerequisites:** complete `flights_agent.ipynb` and `11_dspy_CodeAct` lesson.

By the end, you will:

- Inject domain tools into `PythonInterpreterLocal`
- Wire `dspy.CodeAct` with an `execute_code` wrapper
- Inspect `generated_code` / `code_output` trajectories (not `tool_name` / `tool_args`)
- Run the same book and modify scenarios and verify database side effects
- Maintain multi-turn context with `dspy.History`


## ReAct vs CodeAct

Same seven flight tools — different **interface**.

```mermaid
flowchart TD
  subgraph react [ReAct]
    R_thought[Thought] --> R_tool["Tool call JSON"]
    R_tool --> R_obs[Observation]
    R_obs --> R_thought
  end
  subgraph codeact [CodeAct]
    C_thought[Thought] --> C_code[Python snippet]
    C_code --> C_exec[PythonInterpreterLocal]
    C_exec --> C_out[stdout and state]
    C_out --> C_thought
  end
```

- **ReAct:** the LM chooses **one tool + args** per step. The trajectory records `thought`, `tool_name`, `tool_args`, and `observation`.
- **CodeAct:** the LM writes **executable Python** that can call several injected tools, branch, and loop. The trajectory records `generated_code_N` and `code_output_N`.
- **ReAct wiring:** pass tools directly to `dspy.ReAct(tools=[...])`.
- **CodeAct wiring:** register tools on `PythonInterpreterLocal` so generated code can call `fetch_flight_info(...)`, `book_flight(...)`, and the rest.

## Setup

### Install dependencies

Before starting, install the required packages:

```bash
!pip install -qU dspy pydantic rich
```

### MLflow DSPy Integration

Set up MLflow Tracing to understand what's happening under the hood.

<a href="https://mlflow.org/">MLflow</a> is an LLMOps tool that natively integrates with DSPy and offers explainability and experiment tracking. You can use MLflow to visualize prompts and optimization progress as traces to understand DSPy's behavior better.

![MLflow Trace](../assets/mlflow-tracing-customer-service-agent.png)

1. Install MLflow

```bash
%pip install mlflow>=3.0.0
```

2. Start MLflow UI in a separate terminal

```bash
mlflow ui --port 5000 --backend-store-uri sqlite:///mlruns.db
```

3. Connect the notebook to MLflow

```python
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("DSPy")
```

4. Enable tracing.

```python
mlflow.dspy.autolog()
```

To learn more, visit [MLflow DSPy Documentation](https://mlflow.org/docs/latest/llms/dspy/index.html).

### Import lesson utilities

This exercise reuses `PythonInterpreterLocal` and trajectory helpers from the lesson folder. Add `../lessons` to `sys.path` so those imports resolve from the exercises directory.

In [30]:
import sys
from pathlib import Path

lessons_dir = Path("../lessons").resolve()
sys.path.insert(0, str(lessons_dir))

from ext.python_interpreter import PythonInterpreterLocal
from utils import print_pretty_codeact_trajectory

## Domain scaffold (same as the ReAct exercise)

The airline domain — models, dummy data, and tools — is identical to [flights_agent.ipynb](flights_agent.ipynb). For CodeAct, these tools are **not** passed to the agent module directly; they are registered on the interpreter so generated Python can call them by name.

### Define data structures

In production, this would be the database schema. For this demo, we use [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/) for simplicity.

In [31]:
from pydantic import BaseModel  # https://docs.pydantic.dev/latest/


class Date(BaseModel):
    # LLMs struggle to emit valid `datetime.datetime` values, so we use a simple custom type.
    year: int
    month: int
    day: int
    hour: int


class UserProfile(BaseModel):
    user_id: str
    name: str
    email: str


class Flight(BaseModel):
    flight_id: str
    date_time: Date
    origin: str
    destination: str
    duration: float
    price: float


class Itinerary(BaseModel):
    confirmation_number: str
    user_profile: UserProfile
    flight: Flight


class Ticket(BaseModel):
    user_request: str
    user_profile: UserProfile

### Create dummy data

Create a few flights and users, and initialize empty dictionaries for itineraries and customer support tickets.

In [32]:
user_database = {
    "Adam": UserProfile(user_id="1", name="Adam", email="adam@gmail.com"),
    "Bob": UserProfile(user_id="2", name="Bob", email="bob@gmail.com"),
    "Chelsie": UserProfile(user_id="3", name="Chelsie", email="chelsie@gmail.com"),
    "David": UserProfile(user_id="4", name="David", email="david@gmail.com"),
}

flight_database = {
    "DA123": Flight(
        flight_id="DA123",  # DSPy Airline 123
        origin="SFO",
        destination="JFK",
        date_time=Date(year=2025, month=9, day=1, hour=1),
        duration=3,
        price=200,
    ),
    "DA125": Flight(
        flight_id="DA125",
        origin="SFO",
        destination="JFK",
        date_time=Date(year=2025, month=9, day=1, hour=7),
        duration=9,
        price=500,
    ),
    "DA456": Flight(
        flight_id="DA456",
        origin="SFO",
        destination="SNA",
        date_time=Date(year=2025, month=10, day=1, hour=1),
        duration=2,
        price=100,
    ),
    "DA460": Flight(
        flight_id="DA460",
        origin="SFO",
        destination="SNA",
        date_time=Date(year=2025, month=10, day=1, hour=9),
        duration=2,
        price=120,
    ),
}

# In-memory stores the agent will read from and write to.
itinerary_database = {}
ticket_database = {}

### Define the tools

Every tool function should:

- Have a docstring that describes what the tool does.
- Have type hints on arguments so the LM can generate correctly formatted calls inside Python code.

In [33]:
import random
import string


def fetch_flight_info(date: Date, origin: str, destination: str):
    """Fetch flight information from origin to destination on the given date"""
    flights = []

    for flight_id, flight in flight_database.items():
        if (
            flight.date_time.year == date.year
            and flight.date_time.month == date.month
            and flight.date_time.day == date.day
            and flight.origin == origin
            and flight.destination == destination
        ):
            flights.append(flight)
    if len(flights) == 0:
        raise ValueError("No matching flight found!")
    return flights


def fetch_itinerary(confirmation_number: str):
    """Fetch a booked itinerary information from database"""
    return itinerary_database.get(confirmation_number)


def pick_flight(flights: list[Flight]):
    """Pick up the best flight that matches users' request. we pick the shortest, and cheaper one on ties."""
    sorted_flights = sorted(
        flights,
        key=lambda x: (
            x.get("duration") if isinstance(x, dict) else x.duration,
            x.get("price") if isinstance(x, dict) else x.price,
        ),
    )
    return sorted_flights[0]


def _generate_id(length=8):
    chars = string.ascii_lowercase + string.digits
    return "".join(random.choices(chars, k=length))


def book_flight(flight: Flight, user_profile: UserProfile):
    """Book a flight on behalf of the user."""
    confirmation_number = _generate_id()
    while confirmation_number in itinerary_database:
        confirmation_number = _generate_id()
    itinerary_database[confirmation_number] = Itinerary(
        confirmation_number=confirmation_number,
        user_profile=user_profile,
        flight=flight,
    )
    return confirmation_number, itinerary_database[confirmation_number]


def cancel_itinerary(confirmation_number: str, user_profile: UserProfile):
    """Cancel an itinerary on behalf of the user."""
    if confirmation_number in itinerary_database:
        del itinerary_database[confirmation_number]
        return
    raise ValueError("Cannot find the itinerary, please check your confirmation number.")


def get_user_info(name: str):
    """Fetch the user profile from database with given name."""
    return user_database.get(name)


def file_ticket(user_request: str, user_profile: UserProfile):
    """File a customer support ticket if this is something the agent cannot handle."""
    ticket_id = _generate_id(length=6)
    ticket_database[ticket_id] = Ticket(
        user_request=user_request,
        user_profile=user_profile,
    )
    return ticket_id

In [34]:
def modify_itinerary(confirmation_number: str, new_flight_id: str, new_date: Date):
    """
    Modify an existing itinerary to switch to a new flight (by flight_id) on the given date.
    Returns new confirmation number and the new itinerary.
    """
    # Look up existing itinerary
    itinerary = itinerary_database.get(confirmation_number)
    if itinerary is None:
        raise ValueError(f"Cannot find the itinerary {confirmation_number}")
    user_profile = itinerary.user_profile

    # Search for the requested flight on the given date
    flights = fetch_flight_info(new_date, itinerary.flight.origin, itinerary.flight.destination)
    if not flights or len(flights) == 0:
        raise ValueError(f"No flights found from {itinerary.flight.origin} to {itinerary.flight.destination} on {new_date}")

    # Select the flight with the requested flight_id
    selected_flight = None
    for f in flights:
        if (getattr(f, 'flight_id', None) or (isinstance(f, dict) and f.get('flight_id'))) == new_flight_id:
            selected_flight = f
            break
    if not selected_flight:
        raise ValueError(f"Flight {new_flight_id} not found on {new_date}")

    # Cancel the old itinerary
    cancel_itinerary(confirmation_number, user_profile)

    # Book the new flight
    new_confirmation_number, new_itinerary = book_flight(selected_flight, user_profile)
    return new_confirmation_number, new_itinerary

### Agent signature

Reuse the signature you defined in the ReAct exercise, extended here with a `history` field for multi-turn customer service.

In [35]:
import dspy  # https://dspy.ai/


class DSPyAirlineCustomerService(dspy.Signature):
    """You are an airline customer service agent that helps user book and manage flights.

    Use conversation history to resolve follow-up requests about prior bookings.

    The interpreter exposes: fetch_flight_info, pick_flight, book_flight, fetch_itinerary,
    cancel_itinerary, get_user_info, file_ticket, and modify_itinerary.

    Build dates with keyword arguments, e.g. Date(year=2025, month=9, day=1, hour=0).
    Call fetch_flight_info(date, origin, destination) — date is the first argument.
    Call get_user_info(name) with the passenger name.
    Call book_flight(flight, user_profile) — flight comes first.

    For booking requests, write Python that runs the full workflow in one script:
    get_user_info -> fetch_flight_info -> pick_flight -> book_flight, then print the confirmation number.

    For modification requests, call modify_itinerary(confirmation_number, new_flight_id, new_date).
    """

    user_request: str = dspy.InputField()
    history: dspy.History = dspy.InputField()
    process_result: str = dspy.OutputField(
        desc=(
            "Message that summarizes the process result, and the information users need, e.g., the "
            "confirmation_number if a new flight is booked."
        )
    )

## Create CodeAct Agent

Create the CodeAct agent with `dspy.CodeAct`. You provide:

- A **signature** that defines the task, inputs, and outputs (and documents the interpreter tools).
- A **`PythonInterpreterLocal`** with your domain tools injected.
- An **`execute_code`** wrapper passed as the single tool CodeAct lists to the LM.

The flight tools live on the interpreter (not in CodeAct's `tools` list). Generated Python calls them by name — for example:

```python
user = get_user_info("Adam")
date = Date(year=2025, month=9, day=1, hour=0)
flights = fetch_flight_info(date, "SFO", "JFK")
flight = pick_flight(flights)
confirmation_number, itinerary = book_flight(flight, user)
print(confirmation_number)
```

## Exercise 1: Build the interpreter

Create a `PythonInterpreterLocal` that exposes all seven flight tools and the Pydantic types the generated code needs to construct arguments.

- Add the tools to `namespace`.

In [36]:
flights_interpreter = PythonInterpreterLocal(
    namespace=[
        Date,
        Flight,
        Itinerary,
        Ticket,
        fetch_flight_info,
        fetch_itinerary,
        pick_flight,
        book_flight,
        cancel_itinerary,
        get_user_info,
        file_ticket,
        modify_itinerary # new tool
    ],
)

::: {.note}

Note: `modify_itinerary` was later added because this was seen as a common request. This is why you should track user requests, look at the generated code, and try to refactor it to use higher-level abstractions, which leads to more direct responses, higher accuracy, less tokens, and keeps the cost minimal.

:::

## Exercise 2: Define the `execute_code` wrapper

CodeAct calls this function to run each generated Python snippet. Delegate to the interpreter's `execute` method.

In [37]:
def execute_code(code: str) -> str:
    return flights_interpreter.execute(code)

## Exercise 3: Wire up `dspy.CodeAct`

Create a CodeAct agent that uses your signature, the `execute_code` wrapper, and the interpreter.

- Import `CodeAct` from `dspy.predict`.
- Pass `DSPyAirlineCustomerService` as the signature.
- Pass `execute_code` as the only tool.
- Pass `flights_interpreter` as the interpreter.

In [38]:
from dspy.predict import CodeAct

agent = CodeAct(
    DSPyAirlineCustomerService,
    tools=[execute_code],
    interpreter=flights_interpreter,
)

## Use the Agent

To interact with the agent, pass the user's request through the `user_request` input field defined in your signature.

## Exercise 4: Configure the language model

Select a language model and set up your API key. We use `openrouter/openai/gpt-5-nano` here, but you can change to other models.

For configuration options, see the [DSPy language model guide](https://dspy.ai/learn/programming/language_models/).

In [39]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from pathlib import Path

    if not Path('.env').exists():
        print("No .env file found. Please create one with OPENROUTER_API_KEY.")

    from dotenv import load_dotenv
    load_dotenv(override=True)

print(len(os.environ["OPENROUTER_API_KEY"]))

# Pass the key explicitly...
lm = dspy.LM(
    "openrouter/openai/gpt-5.4-mini",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

dspy.configure(lm=lm)

73


## Conversation history

DSPy does not automatically manage chat state inside modules. To carry context across turns — for example, from a booking to a follow-up modification — use [`dspy.History`](https://dspy.ai/tutorials/conversation_history):

1. Add `history: dspy.History` to your signature (done above).
2. Keep a `dspy.History(messages=[])` instance at runtime.
3. Pass it into every `agent(...)` call.
4. After each turn, append `{"user_request": ..., **result}` to `history.messages`.

See [Managing Conversation History](https://dspy.ai/tutorials/conversation_history) for the full pattern.

In [40]:
conversation_history = dspy.History(messages=[])

### Book a flight

Run the cell below to book a flight for Adam. Note the `confirmation_number` in the output — you will need it in Exercise 6.

In [41]:
booking_request = (
    "please help me book a flight from SFO to JFK on 09/01/2025, my name is Adam"
)

result = agent(user_request=booking_request, history=conversation_history)
conversation_history.messages.append({"user_request": booking_request, **result})

2026/06/18 09:07:28 WARNING dspy.predict.predict: Type mismatch for field 'trajectory': expected str based on given Signature, but the provided value is incompatible: {}.


## Exercise 5: Inspect the CodeAct trajectory

Use `print_pretty_codeact_trajectory` to see the generated Python and interpreter output from the booking above.

- Pass the `result` prediction.
- Pass `result.process_result` as the final answer to display.

In [42]:
print_pretty_codeact_trajectory(result, result.process_result)

📜 generated_code_0:

from datetime import datetime                                                                                      
                                                                                                                   
# Build the date object as requested                                                                               
date = Date(year=2025, month=9, day=1, hour=0)                                                                     
                                                                                                                   
# Full booking workflow                                                                                            
user_profile = get_user_info("Adam")                                                                               
flights = fetch_flight_info(date, "SFO", "JFK")                                                                    
selected_flight = pick_flight(flights)                                                                             
confirmation = book_flight(selected_flight, user_profile)                                                          
                                                                                                                   
print(confirmation)                                                                                                

🖨️ code_output_0: "('34ib1ate', Itinerary(confirmation_number='34ib1ate', user_profile=UserProfile(user_id='1', 
name='Adam', email='adam@gmail.com'), flight=Flight(flight_id='DA123', date_time=Date(year=2025, month=9, day=1, 
hour=1), origin='SFO', destination='JFK', duration=3.0, price=200.0)))"

🤔 reasoning: The booking workflow was completed successfully using the provided itinerary details. The system 
returned a confirmed itinerary with confirmation number 34ib1ate for Adam on a flight from SFO to JFK on 
09/01/2025.

✨ answer: Your flight has been booked successfully. Confirmation number: 34ib1ate. Passenger: Adam. Route: SFO → 
JFK. Date: 09/01/2025.

### Interpret the result

The returned `Prediction` contains:

- `process_result`: the user-facing message defined in your signature.
- `reasoning`: a summary of why the agent took the actions it did.
- `trajectory`: a step-by-step record of the CodeAct loop, including:
  - `generated_code_N`: Python the LM wrote at step N.
  - `code_output_N`: stdout / interpreter feedback from running that code.

Compare this to the ReAct exercise, where the trajectory instead records `thought`, `tool_name`, `tool_args`, and `observation` for each step.

Behind the scenes, `dspy.CodeAct` runs a loop that accumulates code and execution output along with the task description and sends it to the LM until it hits `max_iters` or the task is complete.

## Exercise 6: Modify an existing itinerary

Ask the agent to change Adam's booked flight to DA125 on 09/01.

- Set `confirmation_number` to the value from the booking result above.
- Run the agent with the provided request string.
- `conversation_history` carries the booking turn, so the agent can reference Adam's prior confirmation without re-deriving context from scratch.

In [43]:
confirmation_number = next(iter(itinerary_database))

modify_request = (
    f"i want to take DA125 instead on 09/01, "
    f"please help me modify my itinerary {confirmation_number}"
)

result = agent(user_request=modify_request, history=conversation_history)
conversation_history.messages.append({"user_request": modify_request, **result})

2026/06/18 09:07:31 WARNING dspy.predict.predict: Type mismatch for field 'trajectory': expected str based on given Signature, but the provided value is incompatible: {}.


In [44]:
print_pretty_codeact_trajectory(result, result.process_result)

📜 generated_code_0:

# Modify the existing itinerary 34ib1ate to flight DA125 on 09/01/2025                                             
new_date = Date(year=2025, month=9, day=1, hour=0)                                                                 
result = modify_itinerary("34ib1ate", "DA125", new_date)                                                           
print(result)                                                                                                      

🖨️ code_output_0: "('g1q10wvt', Itinerary(confirmation_number='g1q10wvt', user_profile=UserProfile(user_id='1', 
name='Adam', email='adam@gmail.com'), flight=Flight(flight_id='DA125', date_time=Date(year=2025, month=9, day=1, 
hour=7), origin='SFO', destination='JFK', duration=9.0, price=500.0)))"

🤔 reasoning: The itinerary modification was completed successfully. The original confirmation number 34ib1ate was 
changed to flight DA125 on 09/01/2025, resulting in a new confirmation number g1q10wvt.

✨ answer: Your itinerary has been modified successfully. New confirmation number: g1q10wvt. New flight: DA125. 
Passenger: Adam. Route: SFO → JFK. Date: 09/01/2025.

In [45]:
print(itinerary_database)

{'g1q10wvt': Itinerary(confirmation_number='g1q10wvt', user_profile=UserProfile(user_id='1', name='Adam', email='adam@gmail.com'), flight=Flight(flight_id='DA125', date_time=Date(year=2025, month=9, day=1, hour=7), origin='SFO', destination='JFK', duration=9.0, price=500.0))}


### Inspect conversation-level history

`print_pretty_codeact_trajectory` shows the **CodeAct internal loop** (`generated_code_N`, `code_output_N`).

`dspy.inspect_history()` shows the **conversation-level** multi-turn prompt, including prior `user_request` / `process_result` pairs from `dspy.History`.

In [46]:
dspy.inspect_history(n=1)





[2026-06-18T09:07:34.510221]

System message:

Your input fields are:
1. `user_request` (str): 
2. `history` (History): 
3. `trajectory` (str):
Your output fields are:
1. `reasoning` (str): 
2. `process_result` (str): Message that summarizes the process result, and the information users need, e.g., the confirmation_number if a new flight is booked.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## user_request ## ]]
{user_request}

[[ ## history ## ]]
{history}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## process_result ## ]]
{process_result}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an airline customer service agent that helps user book and manage flights.
        
        Use conversation history to resolve follow-up requests about prior bookings.
        
        The interpreter exposes: fetch_flight_info, pick_flight, book_flight, fetch_itinera

## Conclusion

Congrats on finishing the exercise! You rebuilt the same customer service agent with CodeAct. The key shift from ReAct:

- **Tools** are injected into `PythonInterpreterLocal`, not passed directly to the agent module.
- **`dspy.ReAct`** becomes **`dspy.CodeAct`** plus an `execute_code` wrapper.
- **CodeAct trajectory debugging** uses `print_pretty_codeact_trajectory` (generated Python per agent call).
- **`dspy.History`** carries prior turns across agent calls; append each turn's inputs and outputs after every `agent(...)` call.
- **`dspy.inspect_history()`** shows the multi-turn LM prompt built from conversation history.

## References

- [Build AI Agents with DSPy](https://dspy.ai/tutorials/customer_service_agent/)
- [CodeAct module](https://dspy.ai/api/modules/CodeAct/)
- [Managing Conversation History](https://dspy.ai/tutorials/conversation_history)
- [CodeAct Loop lesson](../lessons/11_dspy_CodeAct.ipynb)